# Plan Missions

This example demonstrates how to account for the deployment and the failures of constellations in a trade study with the `tatc.tradespace` module: the phases of each design's deployment, staged deployments that rules decide on during the mission, simulated failures of satellites and launches restored by policies that are decisions of each design, and the attribution of the spread of the replicates to their uncertain parameters.

It continues the example `SearchTradeSpace`, whose design space and study it imports from `trade_space_example.py` (beside the notebooks). The evaluations of every design are taken from the study's store if `SearchTradeSpace` was run before (or are evaluated again, in a few minutes).

In [1]:
import time
from datetime import timedelta

import pandas as pd

from trade_space_example import ACCURACY, OBJECTIVES, PLAN, START, describe, designs, space, study
from tatc.tradespace import pareto_front, summarize

# the evaluations of every design, the efficient designs (by sufficiency), and the efficient designs by accuracy
results = study.run(designs)
summary = summarize(results, ["satellites", "planes", "sufficient", "accurate"])
summary["description"] = summary.apply(describe, axis=1)
front = summary[pareto_front(summary, OBJECTIVES)].sort_values(["satellites", "planes"])
accuracy = summary[pareto_front(summary, ACCURACY)]

## Account for Deployment

The shares of points sampled so far are those of each design's full constellation, as if its satellites flew for the whole mission. They are launched over months to years, during which fewer satellites sample the points. The launches of each design's deployment divide its mission into phases, in each of which a set of its satellites flies (`Deployment.phases`); each phase is evaluated over the same representative day with the observations of its satellites (`evaluate_phases`, which filters the observations of the full constellation, from the cache), and the phases are combined over the mission (`summarize_phases`): the mean of each share weighted by the duration of the phases, over the five years from the first launch (`sufficient mission`), and the time from the start of development until half of the points are sampled sufficiently (`sufficient time`, infinite if never).

The phased study (a variant of the study with the phases, `study.replace`) evaluates the efficient designs of `SearchTradeSpace` (the most sufficient of each number of satellites and planes), with the years by which each is `deployed`.

In [2]:
from tatc.tradespace import Requirement, aggregate

# a study of the phases of each design's deployment, and of the years until half of the points are sampled sufficiently
phased_study = study.replace(phases=[Requirement("sufficient", ">=", 50)])
PHASED = ["sufficient", "sufficient mission", "sufficient time", "deployed"]
leaders = front.sort_values("sufficient", ascending=False).drop_duplicates(["satellites", "planes"])
phased = phased_study.run(leaders)
deployed = aggregate(phased, PHASED, design_columns=[]).set_index("design")[PHASED]
leaders = leaders.set_index("design")[["description", "satellites", "planes"]].join(deployed)
leaders["lost"] = leaders["sufficient"] - leaders["sufficient mission"]
leaders.sort_values(["satellites", "planes"])[
    ["description", "satellites", "planes", "deployed", "sufficient", "sufficient mission", "lost", "sufficient time"]
].round(1)

,description,satellites,planes,deployed,sufficient,sufficient mission,lost,sufficient time
design,,,,,,,,
163120bf9590,"800 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,3.2,3.2,3.2,0.0,inf
f7413b1f8466,"800 km, 60 deg FOR, Walker 45 deg: 1 x 2 (spac...",2.0,1.0,3.5,16.6,16.6,0.0,inf
33b006848799,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",2.0,2.0,3.8,30.2,27.3,2.9,inf
d5d3fa915ca7,"800 km, 60 deg FOR, Walker 45 deg: 1 x 3 (spac...",3.0,1.0,3.8,38.0,38.0,0.0,inf
1af5f2707be3,"800 km, 60 deg FOR, Walker 45 deg: 1 x 4 (spac...",4.0,1.0,4.5,68.2,63.5,4.7,4.5
6ba50f2632cd,"800 km, 60 deg FOR, Walker 45 deg: 2 x 2 (spac...",4.0,2.0,4.0,69.6,64.3,5.3,4.0
6bef11e40977,"800 km, 60 deg FOR, Walker 75 deg: 2 x 3 (spac...",6.0,2.0,4.5,97.8,87.1,10.7,4.5


Deployment costs the designs of several planes or launches. Six satellites in two planes sample 98% of the points once deployed, but 87% on average over the mission, as the first plane flies alone for three quarters of a year while the second is produced, and they sample half of the points only 4.5 years after development starts, when the second plane is launched. Four satellites in one plane need two launches (of up to three satellites) half a year apart, and lose 5 points of their 68%; four satellites in two planes lose 5 points of their 70%, and two satellites in two planes 3 points of their 30%. The designs launched at once lose nothing. Over a five-year mission, the loss is the share of the mission before full deployment times the shortfall of the partial constellation: a shorter mission, slower production, or fewer launches a year would weigh deployment more, and could reorder the front.

Each phase is evaluated over the same day of July, as if the geometry of the satellites and the season were those of that day throughout the mission (quasi-static phases, see `evaluate_phases`). This misses the drift of the relative nodes of planes of different inclinations or altitudes (which precess at different rates), the drift of the local times of planes that are not sun-synchronous, and seasons. Here, the planes of each design share an altitude and an inclination, which keeps their relative geometry, and the shares do not depend on the sun, so the day should represent the mission. Drawing the start of the day for each replicate over a year (the `epoch` of the design space, which `Observations` and the metrics read, as an `Uncertain` parameter of a design space that extends the other, `DesignSpace.extend`) averages the replicates over the drift and the seasons, at the cost of analyzing the satellites again for each day:

In [3]:
from tatc.tradespace import Uncertain

# the leading designs, with the day of the analysis drawn for each replicate over a year
EPOCH = Uncertain(lambda rng: START + timedelta(days=float(rng.uniform(0, 365))))
epoch_space = space.extend(epoch=EPOCH)
year = phased_study.replace(space=epoch_space).run(epoch_space.designs_of(front, leaders.index))
MEANS = ["sufficient mission", "accurate mission"]
over_year = summarize(year, MEANS).set_index("design")
july = aggregate(phased, MEANS).set_index("design")[MEANS]
leaders[["description", "satellites", "planes"]].join(july.add_suffix(" (July)")).join(
    over_year[[*MEANS, *(f"{m} se" for m in MEANS)]].add_suffix(" (year)")
).sort_values(["satellites", "planes"]).round(1)

,description,satellites,planes,sufficient mission (July),accurate mission (July),sufficient mission (year),accurate mission (year),sufficient mission se (year),accurate mission se (year)
design,,,,,,,,,
163120bf9590,"800 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,3.2,2.8,3.2,3.0,0.7,0.6
f7413b1f8466,"800 km, 60 deg FOR, Walker 45 deg: 1 x 2 (spac...",2.0,1.0,16.6,4.0,16.2,2.0,1.0,0.7
33b006848799,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",2.0,2.0,27.3,30.9,25.8,28.8,1.6,1.6
d5d3fa915ca7,"800 km, 60 deg FOR, Walker 45 deg: 1 x 3 (spac...",3.0,1.0,38.0,11.4,37.8,9.8,0.5,0.7
1af5f2707be3,"800 km, 60 deg FOR, Walker 45 deg: 1 x 4 (spac...",4.0,1.0,63.5,24.5,61.7,22.5,1.8,0.8
6ba50f2632cd,"800 km, 60 deg FOR, Walker 45 deg: 2 x 2 (spac...",4.0,2.0,64.3,59.3,62.1,58.0,1.5,1.3
6bef11e40977,"800 km, 60 deg FOR, Walker 75 deg: 2 x 3 (spac...",6.0,2.0,87.1,71.1,86.7,70.3,0.2,0.8


Over the year, the means of the leading designs are within 2.2 points of those of the day in July (and within three standard errors of the year's means), most a little lower: the one day represents the mission well here. A constellation of planes at different altitudes or inclinations, metrics that depend on daylight, or a field that varies with the seasons would call for drawing the epoch. The phases also assume that a satellite operates until it fails or is replaced: a design life does not retire a satellite whose replacement is not launched, and a planned replacement whose launch fails is not flown again (see `DeploymentSchedule`).

## Stage the Deployment

A deployment need not commit to every plane at once. A stage of a deployment (`Stage`) is a set of planes produced and launched only if a rule decides to, at times during the mission, from what it knows then (a `Decision`: the satellites operating, what they achieve, and the history so far) and from the values of the design, such as its uncertain parameters and its scenario. A stage that is not deployed is neither produced, launched, nor paid for, so that a staged deployment is an option: it saves the cost of a stage where the mission does not need it, and loses what deferring the stage costs where it does (as above, the capability of the satellites not yet launched). Whether and how to stage is a decision of each design (here, a value `staging` of the design space), and the `stages` of a `MissionPlan` are a function of the design.

Here, the design of four satellites most robust to the field whose daily mean is estimated (see `CompareScenarios`: two sun-synchronous planes of two) deploys its second plane at once, never, or only if the field turns out to vary within hours (to decorrelate within 6 hours), as decided at the first launch or a year later. These rules read the field from the scenario, which a mission does not know; the last rule reads what the first plane achieves once it is launched (`Decision.capability`, by the study's metrics), and deploys the second plane if the first samples fewer than half the points accurately. Each staging is evaluated in each field by the phases of its deployment: the mean share of points accurately sampled over the mission, and the present cost:

In [4]:
import dataclasses

from tatc.tradespace import Categorical, Stage

# the fields of CompareScenarios: decorrelating within an hour, in 3 hours, and in 12 hours
FIELDS = {
    "1 hour": {"decorrelation_time": timedelta(hours=1)},
    "3 hours": {},
    "12 hours": {"decorrelation_time": timedelta(hours=12)},
}
STAGING = ["at once", "first plane", "decided at launch", "decided a year later", "measured at launch"]
staging_space = space.extend(staging=Categorical(STAGING))


def stages(d):
    """The stages of a design's deployment: its second plane deployed with the first (at once), never (first plane),
    or if the field decorrelates within 6 hours, as decided at the first launch or a year later, or if the first plane
    accurately samples fewer than half the points, as measured at the first launch."""
    if d.staging == "at once":
        return []
    if d.staging == "first plane":
        return [Stage(1, 0.0, lambda decision: False)]
    if d.staging == "measured at launch":
        return [Stage(1, 0.0, lambda decision: decision.capability["accurate"] < 50)]
    when = 0.0 if d.staging == "decided at launch" else 1.0
    return [Stage(1, when, lambda decision: d.decorrelation_time < timedelta(hours=6))]


# the design of four satellites most robust to the field (see CompareScenarios), with each staging of its deployment, in
# each field
(four,) = space.designs_of(summary[summary.description == "800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00"])
staged = phased_study.replace(
    space=staging_space, plan=dataclasses.replace(PLAN, stages=stages), scenarios=FIELDS
).run([staging_space.design(four, staging=option) for option in STAGING])
staged.groupby(["staging", "scenario"], sort=False)[["accurate mission", "cost present"]].mean().unstack("scenario").round(1)

accurate mission                  cost present          \
scenario                       1 hour 3 hours 12 hours       1 hour 3 hours   
staging                                                                       
at once                          62.6    77.9     91.0        156.4   156.4   
first plane                       4.8    10.0     73.0        118.8   118.8   
decided at launch                62.6    77.9     73.0        156.4   156.4   
decided a year later             49.7    62.8     73.0        154.5   154.5   
measured at launch               62.6    77.9     73.0        156.4   156.4   

                               
scenario             12 hours  
staging                        
at once                 156.4  
first plane             118.8  
decided at launch       118.8  
decided a year later    118.8  
measured at launch      118.8

Deciding at the first launch is worth the most. Where the field varies within hours, the second plane is produced and launched as if it were deployed at once (63% and 78% of the points accurately sampled on average over the mission, for 156 million dollars), and where it varies slowly, it is not, which saves 38 million dollars (24%) for 18 points of the mean (73% against 91%): a trade that the rule makes, and that a mission might not. Deciding a year later costs the fields that need the second plane 13 to 15 points of the mean (50% and 63%), as the first plane flies alone for a year more, and saves only 2 million dollars, by discounting the later production. The value of the option is that of knowing the need when it is exercised: the earlier the field is known, the more staging saves at less loss. The rule that measures makes the same choices as the rule that reads the field, in every field: the first plane alone accurately samples 5% and 10% of the points where the field varies within hours, so that the second plane is deployed, and 73% where it varies over 12 hours, so that it is not. A mission knows what its satellites achieve, not the field, so this is a rule it could follow; a measurement takes time, though (the rule reads at once the capability that the study computes), and one that waits a year to measure loses what deciding a year later loses above. A rule may also read the satellites still operating (see `Decision`), for a reserve plane launched only if satellites fail, or their capability, for one launched only if those in orbit fall short.

## Plan for Failures

A design that needs every satellite to work is fragile, and what failures cost depends on how the mission responds to them. A `FailureModel` gives the life of the satellites from their launch, here a bathtub curve of infant mortality, random failures, and wear-out (`Bathtub` of `Weibull` and `Exponential` lives, illustrative), and the probability that a launch fails. Each replicate simulates the mission of a design with failures drawn from its seed (`Deployment.simulate`, with common random numbers: the satellite at the same position of the same plane fails at the same age in every design), and a `RestorationPolicy` responds to them:

* **degrade**: fly the satellites that remain;
* **produce**: restore each failed satellite by producing a new one (after half a year of procurement, on the production line) and launching it;
* **spare in orbit**: a spare in each plane, launched with it, takes a failed satellite's place within months (others are produced), and is produced again;
* **spare on ground**: a spare, produced after the first satellites and stored, is launched at the next slot (others are produced), and is produced again.

The policy is a decision of each design: a fragment of the design space (`policy_space`) decides whether to restore failed satellites, and the spares in orbit (in each plane) and on the ground (here, none or one of each, and none without restoration, which would never use them), and configures the policy of each design (from a base policy of half a year of procurement and spares that are produced again). The efficient designs by accuracy, each with each of the five policies, form a design space of both (one that extends the other with the policy, `DesignSpace.extend`), in which each history gives both the cost of what is produced and launched (the restorations and spares, and the launches that fail) and the phases in which its satellites operate, evaluated as in the previous section: the share of points accurately sampled on average over the mission, and in the worst 10% of the histories, and the cost at 70% confidence.

The failures are cheap to simulate, but each phase's share is expensive to evaluate, and most phases repeat the same sets of operating satellites. Each replicate (a sample of points, the expensive part) therefore simulates 20 histories of the failures of its mission (`Failures` of the study, with `Deployment.simulate_many`, stratified: a Latin hypercube over the histories in which each satellite's life and each launch draws from its own stratum of its distribution, which estimates means several times more precisely than independent histories), evaluates them together (`evaluate_missions`) with a cache that evaluates each set of satellites once (`PhaseCache`, shared by the histories, policies, and designs of a replicate), and summarizes them (`summarize_missions`): their mean share, and their worst 10% (the risk of the mission), and their cost at 70% confidence. Five replicates of 20 histories measure the risk of each mission with 100 histories, at a fraction of the cost of 100 replicates. Without restoration (degrading), the satellites fail independently, so that the mean share over the failures is computed exactly (`expected_missions`, from the probability that each set of satellites operates), without sampling error; its worst 10% and cost still come from the histories.

In [5]:
from tatc.tradespace import (
    Bathtub,
    Exponential,
    FailureModel,
    Failures,
    Integer,
    RestorationPolicy,
    Weibull,
    policy_space,
)

# illustrative lives: about 9% fail in their first year, 3% a year at random, and wear-out after about 8 years
FAILURE_MODEL = FailureModel(
    life=Bathtub(infant=Weibull(shape=0.5, scale=100), random=Exponential(0.03), wear_out=Weibull(shape=4, scale=9)),
    launch=0.05,
)
print(f"{1 - FAILURE_MODEL.life.survival(5):.0%} of the satellites fail in 5 years")
BASE_POLICY = RestorationPolicy(procurement_years=0.5, replenish=True)
POLICY = policy_space(orbit_spares=Integer(0, 1), ground_spares=Integer(0, 1), replenish=True)

# the design space with a policy, and the efficient designs by accuracy with each policy
policy_trade = space.extend(policy=POLICY)
with_policies, base_of = [], {}
for base in space.designs_of(accuracy):
    for policy in POLICY.enumerate():
        design = policy_trade.design(base, policy=policy)
        with_policies.append(design)
        base_of[design.key] = base.key


def describe_policy(row):
    """A short description of a policy."""
    if not row["policy.restore"]:
        return "degrade"
    orbit, ground = row["policy.orbit_spares"] > 0, row["policy.ground_spares"] > 0
    return {
        (False, False): "produce",
        (True, False): "spare in orbit",
        (False, True): "spare on ground",
        (True, True): "spares in orbit and on ground",
    }[orbit, ground]


# 20 histories of each replicate, restored by the policy of each design, with the mean share accurately sampled over
# the mission in the worst 10% of the histories, and the cost at 70% confidence (over the histories)
FAILURES = Failures(
    FAILURE_MODEL, BASE_POLICY, histories=20, seed="failure_seed", quantiles={"accurate mission": 0.1, "cost": 0.7}
)
failures_study = study.replace(space=policy_trade, failures=FAILURES)
began = time.perf_counter()
trades = failures_study.run(with_policies)
print(f"{len(with_policies)} designs x 5 replicates x 20 histories in {time.perf_counter() - began:.0f} s")
# the means over the replicates of the histories' mean share, worst 10%, and cost at 70% confidence
ROBUST = {"accurate": "accurate mission", "accurate p10": "accurate mission histories p10", "cost p70": "cost histories p70"}
traded = aggregate(trades, statistics=ROBUST).set_index("design")
traded["policy"] = traded.apply(describe_policy, axis=1)
traded["base"] = [base_of[key] for key in traded.index]
traded = traded[[*ROBUST, "policy", "base"]]

# the shares of the full constellations, without deployment or failures, for comparison
nominal = study.replace(replicates=20).run(accuracy)
full = aggregate(nominal, ["accurate"]).set_index("design")["accurate"]
descriptions = accuracy.set_index("design")["description"]
ORDER = ["degrade", "produce", "spare in orbit", "spare on ground", "spares in orbit and on ground"]
table = traded.pivot(index="base", columns="policy", values=list(ROBUST))
table = table[[(metric, policy) for metric in ROBUST for policy in ORDER]]
table.insert(0, ("full constellation", ""), full.reindex(table.index))
table.insert(0, ("description", ""), descriptions.reindex(table.index))
table.sort_values(("full constellation", "")).round(1)

37% of the satellites fail in 5 years


65 designs x 5 replicates x 20 histories in 10 s


description  \
policy                                                            
base                                                              
9b3481aea973                800 km, 60 deg FOR, SSO: 1 at 12:00   
fac5d43a19c7                800 km, 60 deg FOR, SSO: 2 at 06:00   
b85daf65c3a4  800 km, 60 deg FOR, Walker 75 deg: 1 x 3 (spac...   
104e33a3c842  800 km, 60 deg FOR, Walker 75 deg: 1 x 4 (spac...   
33b006848799  800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...   
c576e2e2dbd1    800 km, 60 deg FOR, SSO: 2 at 15:00, 1 at 09:00   
f216ec70e160  800 km, 60 deg FOR, Walker 60 deg: 3 x 1 (spac...   
3690f1194419    800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00   
cb6bd361b577  800 km, 60 deg FOR, Walker 75 deg: 4 x 2 (spac...   
716cb479efda  700 km, 60 deg FOR, Walker 75 deg: 3 x 2 (spac...   
5529c21520be  800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spac...   
679ca275c8fc  800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spac...   
de505bc89299  800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spac...   

             full constellation accurate                         \
policy                           degrade produce spare in orbit   
base                                                              
9b3481aea973                3.4      3.2     3.9            4.0   
fac5d43a19c7               10.0      7.0     9.2            9.6   
b85daf65c3a4               16.7     10.9    14.8           14.7   
104e33a3c842               29.3     16.1    22.3           24.4   
33b006848799               33.0     17.7    26.1           28.2   
c576e2e2dbd1               57.0     33.1    47.3           50.7   
f216ec70e160               60.4     32.9    44.9           48.0   
3690f1194419               82.8     50.7    68.5           68.8   
cb6bd361b577               96.0     75.8    81.2           76.3   
716cb479efda               97.0     64.2    78.0           74.2   
5529c21520be               98.3     79.3    82.8           76.8   
679ca275c8fc               98.3     79.5    83.0           76.8   
de505bc89299               98.3     79.5    82.9           76.9   

                                                           accurate p10  \
policy       spare on ground spares in orbit and on ground      degrade   
base                                                                      
9b3481aea973             4.1                           4.0          0.4   
fac5d43a19c7             9.8                           9.7          2.1   
b85daf65c3a4            16.0                          14.9          4.6   
104e33a3c842            24.4                          24.9          6.1   
33b006848799            28.9                          28.5          0.6   
c576e2e2dbd1            51.7                          51.2          7.1   
f216ec70e160            48.1                          48.3          9.5   
3690f1194419            72.5                          69.3         10.8   
cb6bd361b577            81.5                          76.2         60.4   
716cb479efda            79.6                          74.3         38.9   
5529c21520be            82.8                          76.9         66.4   
679ca275c8fc            83.0                          76.8         69.0   
de505bc89299            83.0                          76.9         68.0   

                                                     \
policy       produce spare in orbit spare on ground   
base                                                  
9b3481aea973     3.5            3.7             3.7   
fac5d43a19c7     7.3            8.3             8.6   
b85daf65c3a4    12.2           12.9            14.3   
104e33a3c842    18.0           20.4            21.0   
33b006848799    19.5           25.1            25.6   
c576e2e2dbd1    38.5           44.7            44.9   
f216ec70e160    37.7           43.0            42.4   
3690f1194419    56.2           61.3            62.2   
cb6bd361b577    73.8           69.8            73.9   
716cb479efda    70.1           67.3          

The 6,500 histories (100 of each of the 65 designs and policies) take about ten seconds here. The shares accurately sampled over the mission fall short of those of the full constellation under every policy, by the phases of deployment and the failures of launches as well as of satellites (37% of which fail within five years), but for the single satellite, whose shares differ by less than the sampling error of five replicates (the full constellation's are of 20). The response to failures matters most for the designs of several satellites in several planes, and in their worst histories:

* **Degrading** is cheapest, but loses 12 to 18 points of the mean against producing replacements for the designs of three to six satellites in two or three planes, and more in their worst 10% of histories (39% against 70% for six satellites in three Walker planes; 1% against 20% for two satellites in two planes, which can lose both). Nine satellites in three planes lose only 3.5 points: a failure leaves eight. The means of degrading are exact (`expected_missions`); those of the other policies are estimates from 100 histories.
* **Producing replacements** restores most of it, at 17 to 34% more cost at 70% confidence than degrading.
* **Spares** cost 28 to 56% more than degrading (up to 62% for spares both in orbit and on the ground), and pay off in the worst histories of the small designs of several planes: two satellites in two planes keep about 25% in their worst histories with a spare, against 20% by producing replacements. A spare on the ground costs less than a spare in each plane for the designs of several planes, and restores as well or better.
* **A spare in orbit can hurt**: for six to nine satellites in three or four planes, the spares delay the deployment (each plane waits for its spare to be produced, and a fourth satellite in a plane of three takes a second launch), so that they sample less on average (74% against 78% for six satellites, 76% against 81% for eight, 77% against 83% for nine) at a higher cost. Spares in orbit and on the ground together cost the most, and sample about as much as a spare on the ground alone, or less.

In [6]:
# the efficient designs and policies, by their cost and their accuracy in the worst 10% of the histories
efficient = traded[pareto_front(traded, {"cost p70": "min", "accurate p10": "max"})]
efficient.assign(description=descriptions.reindex(efficient.base).to_numpy()).sort_values("cost p70")[
    ["description", "policy", "accurate", "accurate p10", "cost p70"]
].round(1)

,description,policy,accurate,accurate p10,cost p70
design,,,,,
2a310c00151b,"800 km, 60 deg FOR, SSO: 1 at 12:00",degrade,3.2,0.4,110.2
bc2076a0d3d8,"800 km, 60 deg FOR, SSO: 2 at 06:00",degrade,7.0,2.1,128.9
bb6574900ac3,"800 km, 60 deg FOR, SSO: 1 at 12:00",produce,3.9,3.5,130.6
638697713902,"800 km, 60 deg FOR, Walker 75 deg: 1 x 3 (spac...",degrade,10.9,4.6,146.2
43f50c8bc38a,"800 km, 60 deg FOR, SSO: 2 at 06:00",produce,9.2,7.3,153.4
ad99c42082ab,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",produce,26.1,19.5,159.4
cc2e14e787ff,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",spare on ground,28.9,25.6,175.2
eeb7c70a720b,"800 km, 60 deg FOR, SSO: 2 at 15:00, 1 at 09:00",produce,47.3,38.5,198.1
3a902b976460,"700 km, 60 deg FOR, Walker 75 deg: 3 x 2 (spac...",degrade,64.2,38.9,209.5


The front of cost at 70% confidence against the share accurately sampled in the worst 10% of the histories chooses designs and policies together. Along it, degrading is the cheapest policy of a design and restoring buys robustness: two satellites in two Walker planes keep 20% of the points accurately sampled in their worst histories if they produce replacements (159 million) and 26% with a spare on the ground (175 million); six satellites in three Walker planes keep 39%, 70%, and 72% if they degrade, produce replacements, or keep a spare on the ground, for 210, 279, and 294 million. Nine satellites in three planes degrade gracefully (69% in their worst histories for 254 million), and producing replacements raises them to 79% for 340 million. A spare on the ground is efficient for designs of two to six satellites, but a spare in orbit is not efficient for any design here, as it delays the deployment. The policy that a design deserves depends on the design, which is why the policy is a decision of the design rather than of the study.

## Find What Spreads the Replicates

The replicates of a design differ by its uncertain parameters: here, the sample of points (`points_seed`), the histories of failures (`failure_seed`), and the growth of the cost of development (`cost_growth`, see `TradeCostAndUtility`). Which of them spreads an objective says what would sharpen it: the spread from the sample of points and from the histories is noise of the estimates, which more points or more histories reduce, and that from the growth of costs is a risk of the mission. `attribute_variance` finds it by evaluating each design again with one parameter drawn as in other replicates and the others held, which works for parameters whose values do not relate to the outcomes in any order, such as seeds. The total share of a parameter is the share of the variance between replicates that it causes, alone or with others (near 0 if it does not matter), with its standard error (`se`, by a bootstrap of the replicates); its first share (`summary("first")`) is the share that knowing it would remove. Here, for the efficient designs and policies of the study of failures, from 20 replicates, 20 more, and 20 with each parameter drawn as in the second 20:

In [7]:
from tatc.tradespace import attribute_variance

# the shares (%) of the variance between the replicates of the efficient designs and policies that each uncertain
# parameter causes, alone or with others, from 20 replicates, 20 more, and 20 with each parameter drawn as in the
# second 20
began = time.perf_counter()
attribution = attribute_variance(
    failures_study, list(ROBUST.values()), policy_trade.designs_of(trades, efficient.index), replicates=20
)
print(f"{len(attribution.results)} evaluations in {time.perf_counter() - began:.0f} s")
attribution.summary().mul(100).round(1)

1700 evaluations in 50 s


,accurate mission,accurate mission histories p10,cost histories p70,accurate mission se,accurate mission histories p10 se,cost histories p70 se
parameter,,,,,,
points_seed,92.4,30.4,0.0,6.4,3.3,0.0
failure_seed,3.6,67.0,35.8,0.7,6.5,7.1
cost_growth,0.0,0.0,81.8,0.0,0.0,19.0


The attribution takes 1,700 evaluations (85 of them those of the study above), under a minute here. The sample of points spreads the mean shares accurately sampled (92% of their variance between replicates), and the failures hardly do (4%): the means of degrading are exact, and those of the other policies average 20 histories. The share in the worst 10% of the histories is spread mostly by the failures (67%, and 30% by the points), as a quantile of 20 histories is a noisy estimate. Both are noise of the estimates rather than risks of the missions, and they call for different remedies: more points would sharpen the means, and more histories in each replicate the worst 10%, which is cheap (the 6,500 histories above take about ten seconds). The cost at 70% confidence of a replicate is spread by the growth of the cost of development (82%), a risk of the mission, and by the failures (36%), which cost replacements and spares for the policies that restore satellites; these shares are the least certain (standard errors of 19 and 7 points). The total shares of each objective sum to about 1 within their errors, so the parameters hardly interact.

The order of designs on the front is settled by their differences, in which the parameters that act alike on two designs cancel (with common random numbers, the designs of a replicate share its parameters). `differences` attributes the variance of the differences between pairs of designs, from the same evaluations: here, between the designs and policies adjacent along the front, by cost at 70% confidence:

In [8]:
# the shares (%) of the variance of the differences between the efficient designs and policies adjacent along the
# front (by cost), from the same evaluations
ordered = efficient.sort_values("cost p70").index
attribution.differences(list(zip(ordered, ordered[1:]))).summary().mul(100).round(1)

,accurate mission,accurate mission histories p10,cost histories p70,accurate mission se,accurate mission histories p10 se,cost histories p70 se
parameter,,,,,,
points_seed,92.1,30.4,0.0,6.4,4.5,0.0
failure_seed,2.9,73.0,105.8,0.5,6.5,4.7
cost_growth,0.0,0.0,0.0,0.0,0.0,0.0


The growth of the cost of development cancels in every difference: all of these designs fly the same imager, whose development grows alike, so that it is a risk of each mission but not a reason to prefer one of them to another, and the failures alone unsettle the order of their costs (all of the variance of the differences, 106 ± 5%). The shares accurately sampled differ by the parameters that spread them: by the sample of points for the means (92%), and mostly by the histories for the worst 10% (73%, and 30% by the points). More points would settle the order of neighbors by their means, and more histories their order by the worst 10% and by cost.

## Summary

* A `DeploymentSchedule` plans each design's deployment from rates of production and launch, which divides its mission into phases of the satellites that fly, evaluated with the observations of the full constellation (over one day, which an epoch drawn for each replicate shows to represent the mission here) for the mean capability over the mission and the time until a requirement is met.
* A `Stage` of a deployment is produced and launched only if a rule decides to, from what is known then (a `Decision`) and the values of the design: an option whose value the analyses of a study measure (here, deciding at the first launch whether the field needs a second plane saves 38 million dollars where it does not, at no loss where it does, and deciding a year later loses 13 to 15 points of the mean where it does). A rule may read what the satellites in orbit achieve (`Decision.capability`): here, a second plane deployed if the first accurately samples fewer than half the points makes the same choices without knowing the field.
* Simulated missions draw the failures of satellites and launches from lives whose failure rates vary over the lifecycle (`FailureModel`), and restore them or not (`RestorationPolicy`, by spares in orbit or on the ground, or by producing new ones), for both the cost and the capability of each policy (exactly, without sampling, for missions that do not restore failed satellites); the policy is a decision of each design (`policy_space`), on the front with the design.
* `attribute_variance` finds the uncertain parameters that spread the replicates of each design (by drawing each again with the others held): here, the sample of points spreads the mean shares and the histories of failures their worst 10%, which says what would sharpen each, and the growth of costs spreads the costs of designs alike, so that it cancels in their differences (`Attribution.differences`) and does not change their order.